# PerconAI - Master MOLAB Execution Notebook
This single notebook executes the entire pipeline from Day 1 to Phase 4 on the MOLAB cluster.


In [ ]:
# 0. UNZIP PERCONAI.ZIP AND SETUP ENVIRONMENT
import os
import sys
import zipfile

# Unzip the uploaded PerconAI.zip into the current directory
if os.path.exists("PerconAI.zip"):
    print("Extracting PerconAI.zip...")
    with zipfile.ZipFile("PerconAI.zip", "r") as zip_ref:
        zip_ref.extractall(".")

# Fix Path in case you are running this from inside the 'notebooks' directory
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..')) if 'notebooks' in current_dir else current_dir
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
import numpy as np
import pandas as pd
import psutil

In [ ]:
# 1. HUGGING FACE SETUP
import os
os.environ["HF_TOKEN"] = "<HF_TOKEN_REMOVED>"
HF_REPO = "Satabarto/Percon2"

In [ ]:
# 2. RUN DAYS 1-3: Setup and Base Model
try:
    from validate_day1 import run_day1_checkpoint
    run_day1_checkpoint()
except ImportError:
    print("Day 1 script not found, assuming pre-validated.")

In [ ]:
# 3. RUN DAY 4: Actions API
from validate_day4 import run_day4_checkpoint
run_day4_checkpoint()

In [ ]:
# 4. RUN DAY 5: Energy Profiler
from validate_day5 import run_day5_checkpoint
run_day5_checkpoint()

In [ ]:
# 5. RUN DAY 6: Drift Regimes
from validate_day6 import run_day6_checkpoint
run_day6_checkpoint()

In [ ]:
# 6. RUN DAY 7: Proxy Triggers
from validate_day7 import run_day7_checkpoint
run_day7_checkpoint()

In [ ]:
# 7. RUN DAY 8: Hindsight Oracle
from validate_day8 import run_day8_checkpoint
run_day8_checkpoint()

In [ ]:
# 8. RUN DAY 9: JADE Predictor Engine
from validate_day9 import run_day9_checkpoint
run_day9_checkpoint()

In [ ]:
# 9. PHASE 4: PREPARE REAL DATA
from scripts.prepare_real_data import main as prep_main
import sys
# Override sys.argv to pass arguments to the script
sys.argv = [
    "prepare_real_data.py",
    "--dataset", "UCI-HAR",
    "--output_dir", "/tmp/jade_data",
    "--hf_repo", HF_REPO,
    "--hf_token", os.environ["HF_TOKEN"]
]
prep_main()

In [ ]:
# 10. PHASE 4: GENERATE META-TRAINING CORPUS
from scripts.generate_corpus import main as gen_main
import sys
# Run generation for Regime R1 (Example)
sys.argv = [
    "generate_corpus.py",
    "--dataset", "UCI-HAR",
    "--regime", "r1_sudden_subject",
    "--subject_idx", "1",
    "--output_dir", "/tmp/jade_data",
    "--hf_repo", HF_REPO,
    "--hf_token", os.environ["HF_TOKEN"]
]
gen_main()

# 10.5 PHASE 4: AGGREGATE CORPUS
Combine chunks into the frozen meta-corpus.

In [ ]:
from scripts.aggregate_corpus import main as agg_main
import sys
import os
sys.argv = [
    "aggregate_corpus.py",
    "--input_dir", "/marimo/jade_data/corpus",
    "--output_dir", "data/processed",
    "--hf_repo", HF_REPO,
    "--hf_token", os.environ["HF_TOKEN"]
]
agg_main()

# 11. DAY 13-15: JADE PREDICTOR TRAINING
Train the meta-controller on the generated corpus.

In [ ]:
from scripts.train_predictor import main as train_pred_main
from scripts.train_fleet_jade import main as train_fleet_main
import sys

sys.argv = [
    "train_predictor.py",
    "--split", "indomain",
    "--model_type", "logistic"
]
train_pred_main()

sys.argv = ["train_fleet_jade.py"]
train_fleet_main()

# 12. DAY 16: FAST EVALUATION SANITY CHECK
Test the pipeline end-to-end on a fast subset.

In [ ]:
from scripts.eval_fast import main as eval_fast_main
import pandas as pd
import sys

sys.argv = [
    "eval_fast.py",
    "--n_subjects", "2",
    "--n_seeds", "1",
    "--output_dir", "tmp_eval_fast"
]
eval_fast_main()

fast_results = pd.read_parquet("tmp_eval_fast/results_fast.parquet")
display(fast_results.groupby("policy")[["stream_acc", "adaptation_tax", "joule_regret"]].mean())

# 13. DAY 17: FULL EVALUATION (MOLAB ARRAY)
Generate manifest and run full eval.

In [ ]:
from scripts.generate_task_manifest import main as manifest_main
from scripts.eval_worker import main as eval_worker_main
from scripts.aggregate_eval_results import main as agg_eval_main
from pathlib import Path
import sys

sys.argv = ["generate_task_manifest.py"]
manifest_main()

# Note: Running all tasks linearly takes hours! Running just 2 for testing.
Path("data/results/raw_eval_chunks").mkdir(parents=True, exist_ok=True)
for task_id in range(2):
    sys.argv = [
        "eval_worker.py",
        "--task_id", str(task_id),
        "--manifest_path", "data/results/task_manifest.csv",
        "--output_dir", "data/results/raw_eval_chunks"
    ]
    eval_worker_main()

sys.argv = ["aggregate_eval_results.py"]
agg_eval_main()

# 14. DAY 18-19: STATS & VISUALIZATION
Generate figures and tables natively.

In [ ]:
from scripts.analyze_results import main as analyze_main
from scripts.generate_figures import main as figures_main
from scripts.generate_tables import main as tables_main
import sys

sys.argv = ["analyze_results.py"]
analyze_main()

sys.argv = ["generate_figures.py"]
figures_main()

sys.argv = ["generate_tables.py"]
tables_main()

from IPython.display import Image, display
display(Image(filename='figures/fig3_pareto.png'))
display(Image(filename='figures/fig5_breakeven.png'))